# trio-retina — from Supervision: pipe your `sv.Detections` straight in

**What you'll see:** an existing Roboflow Supervision detection object flows through `Detection.from_supervision(...)` → `ZoneRule` and out comes `zone.enter` / `zone.dwell` events.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machinefi/trio-retina/blob/main/notebooks/03_from_supervision.ipynb)


## No extra deps needed

`Detection.from_supervision` is **duck-typed** — Retina never imports `supervision`, it just reads `.xyxy` / `.confidence` / `.class_id` / `.data`. So this demo needs **only `trio-retina`**; we fake the `sv.Detections` object with a `SimpleNamespace`. In real use you pass your actual `sv.Detections` straight in (with `supervision` installed).

In [ ]:
%pip install trio-retina

## A fake `sv.Detections` + the adapter

`fake_sv_detections` mirrors Supervision's attribute shape exactly. The adapter turns each frame's detection object into Retina `Detection`s via `from_supervision`.

In [ ]:
from types import SimpleNamespace

import numpy as np
from retina import IoUTracker, Retina, Zone, ZoneRule
from retina.detect import Detection


def fake_sv_detections(x: float):
    """Stand-in for sv.Detections: one 'person' box at horizontal pos x."""
    return SimpleNamespace(
        xyxy=np.array([[x - 10, 40, x + 10, 60]], dtype=float),
        confidence=np.array([0.9]),
        class_id=np.array([0]),
        data={"class_name": np.array(["person"])},
    )


class SupervisionAdapter:
    """Each frame: run your Supervision pipeline, hand its output to from_supervision."""

    def __init__(self):
        self.f = 0

    def __call__(self, frame: np.ndarray) -> list[Detection]:
        x = min(self.f * 6, 50)  # walk in, then linger to trigger dwell
        self.f += 1
        sv_dets = fake_sv_detections(x)
        # In real use: sv_dets = your_supervision_pipeline(frame)
        return Detection.from_supervision(sv_dets)

## Run it → tracked `zone.enter` / `zone.dwell`

The person enters the dock and lingers; with `dwell_s=3.0` that produces a `zone.dwell` event. Output is the same standard `retina.event` JSON as every other detector.

In [ ]:
dock = Zone("dock", [(0.4, 0), (0.6, 0), (0.6, 1), (0.4, 1)], normalized=True)

cam = Retina(
    source_id="cam_01",
    detector=SupervisionAdapter(),
    tracker=IoUTracker(min_hits=2),
    rules=[ZoneRule(dock, classes={"person"}, dwell_s=3.0)],
)

frames = [(np.zeros((100, 100, 3), np.uint8), float(i)) for i in range(18)]
for event in cam.run(frames):
    print(event.to_json())